# Day 4b — Generative AI and the Gemini API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tech4alltraining/aiml/blob/main/mlai-genai-internship/notebooks/05_day4b_genai_api.ipynb)

**ML/AI & GenAI Internship** · Needs a free Gemini API key. Each concept: examples → 5 exercises → quiz → tasks.

---

## What this notebook covers

Four concepts, each in the same shape: **📘 Examples → ✏️ 5 exercises → 🧠 Quiz → 🎯 Tasks.**

| # | Concept |
|---|---|
| 1 | Your first API call, and what comes back |
| 2 | Prompt engineering: the five parts and the four types |
| 3 | Temperature, top-p and top-k |
| 4 | Structured output and conversation memory |

> ⚠️ **This notebook ships without saved outputs**, because every cell calls a live API with **your** key. Run them yourself — that is the point.

**You need a free Gemini API key:** [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

In [ ]:
# Install the SDK. On Colab this is needed every session, because the
# runtime resets when it disconnects.

!pip install -q -U google-genai

In [ ]:
# Enter your key. getpass HIDES what you type, so the key is never
# saved inside this notebook file.
#
# NEVER paste a key directly into a cell. Notebook outputs and source are
# saved together, so the key would travel with the file when you share it
# or commit it to GitHub.

import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

print("Key loaded:", "GEMINI_API_KEY" in os.environ)

In [ ]:
# ALTERNATIVE 1 - Colab Secrets (better: store it once, reuse forever)
#   1. Click the key icon in the left sidebar
#   2. Add a secret named GEMINI_API_KEY, paste the value
#   3. Toggle "Notebook access" on
#
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# ALTERNATIVE 2 - on your own machine, a .env file
#
# from dotenv import load_dotenv
# load_dotenv()

print("Both methods are explained in ../setup-guide.md")

In [ ]:
from google import genai
from google.genai import types
import json

MODEL_NAME = "gemini-3.5-flash"

# The client is your connection to the API. Create it once and reuse it.
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def ask(prompt, **config):
    """Small helper so later cells stay short and readable."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(**config) if config else None,
    )
    return response.text

print("Client ready. Model:", MODEL_NAME)

---

# Concept 1 — Your first API call

🧠 **Analogy: ordering food by phone.**

An API is a way for your program to ask another program for something. You do not need to know how the kitchen works — you send a request in an agreed format, and food arrives.

```text
Your Python  ->  request (prompt + settings)  ->  Google's servers
                                                       |
Your Python  <-  response (text + metadata)   <--------+
```

Two things to understand from the start:

1. **You are billed per token**, both for what you send and what comes back
2. **The connection is stateless** — the server remembers nothing between calls

## 📘 Examples

### Example 1 — the simplest possible call

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain overfitting to a first-year student in exactly three sentences.",
)

print(response.text)

### Example 2 — what actually came back

In [ ]:
# The response is an ordinary object, not magic. Look inside it.

print("--- The text ---")
print(response.text[:300])

print("\n--- The bill (this is what you are charged on) ---")
usage = response.usage_metadata
print(f"Prompt tokens : {usage.prompt_token_count}")
print(f"Output tokens : {usage.candidates_token_count}")
print(f"Total tokens  : {usage.total_token_count}")

print("\n--- Why did it stop? ---")
print("finish_reason:", response.candidates[0].finish_reason)
print("  STOP = it finished naturally")
print("  MAX_TOKENS = it was cut off, raise max_output_tokens")
print("  SAFETY = a safety filter blocked it")

> **The JSON Treasure Hunt.** Print the whole `response` object once. You will get a wall of nested data. Find three things in it: the generated text, the total token count, and the safety ratings.
>
> **This demystifies the whole thing.** An LLM response is not a thinking machine's thoughts — it is a standard software system returning a predictable, nested dictionary.

In [ ]:
# Try it: print the entire object. Read it once, then never again.
print(response)

### Example 3 — errors you will actually hit

In [ ]:
# Real code must handle failure. These are the three you will meet.

try:
    reply = ask("Say hello in exactly five words.")
    print("OK:", reply.strip())
except Exception as error:
    name = type(error).__name__
    print(f"{name}: {error}")
    print()
    print("The three common causes:")
    print("  400 / API key not valid   -> wrong or revoked key, or stray spaces")
    print("  429 RESOURCE_EXHAUSTED    -> free-tier rate limit; wait a minute")
    print("  KeyError GEMINI_API_KEY   -> the key never got loaded above")

## ✏️ Practice now — First API call

Five short exercises. Do them **before** moving on — this is where the learning happens.

**1.** Ask the model to explain **cross-validation** in two sentences. Print the answer.

**2.** Print the total token count for that call.

**3.** Ask the same question but add `max_output_tokens=20`. What happens? Check `finish_reason`.

**4.** Ask a question in a language other than English. Does the token count go up for the same word count?

**5.** Write a loop that asks three different questions and prints each answer's token count and cost at $0.30 per million tokens.

In [ ]:
# 1.


# 2.


# 3.


# 4.


# 5.

<details>
<summary><b>Solutions — open only after you have tried all five</b></summary>

```python
# 1
r = client.models.generate_content(model=MODEL_NAME,
        contents="Explain cross-validation in two sentences.")
print(r.text)

# 2
print("tokens:", r.usage_metadata.total_token_count)

# 3
r2 = client.models.generate_content(model=MODEL_NAME,
        contents="Explain cross-validation in two sentences.",
        config=types.GenerateContentConfig(max_output_tokens=20))
print(r2.text)
print("finish_reason:", r2.candidates[0].finish_reason)   # MAX_TOKENS - cut off

# 4
r3 = client.models.generate_content(model=MODEL_NAME,
        contents="क्रॉस-वैलिडेशन को दो वाक्यों में समझाइए।")
print(r3.usage_metadata.total_token_count)
# Non-English usually costs MORE tokens for the same meaning.

# 5
for q in ["What is a p-value?", "What is recall?", "What is a token?"]:
    r = client.models.generate_content(model=MODEL_NAME, contents=q)
    t = r.usage_metadata.total_token_count
    print(f"{t:>5} tokens  ${t / 1_000_000 * 0.30:.6f}  {q}")
```

</details>

## 🧠 Quick quiz — First API call

Answer all five from memory. No scrolling back.

**Q1.** What does `usage_metadata` tell you and why does it matter?

**Q2.** Your response is cut off mid-sentence. What do you check, and what do you fix?

**Q3.** You get `429 RESOURCE_EXHAUSTED`. What happened?

**Q4.** Why should you never paste your API key directly into a notebook cell?

**Q5.** Someone says 'the model remembers our earlier conversation'. Are they right?

<details>
<summary><b>Answers</b></summary>

**A1.** The number of tokens in your prompt, in the output, and in total. It matters because **that is what you are billed on** — a long prompt costs money on every single call, not once.

**A2.** Check `response.candidates[0].finish_reason`. If it says `MAX_TOKENS`, raise `max_output_tokens`. If it says `SAFETY`, your prompt triggered a content filter.

**A3.** You exceeded the free-tier rate limit. Wait a minute. If you are calling the API inside a loop, add a pause or reduce the iterations — this is the most common way students hit it.

**A4.** Notebook source and outputs are saved together in the `.ipynb` file. The key would travel with the file when you share it or push it to GitHub, where automated scanners find keys within minutes. Use `getpass` or Colab Secrets.

**A5.** No. The API is **stateless** — each call starts blank. If a chatbot appears to remember, the *program* is re-sending the whole conversation history with every request.

</details>

## 🎯 Tasks — First API call

Longer work. Do these after the session, in your own time.

### Task 1: The cost calculator

Build a small tool that, for any prompt, reports:

1. Estimated tokens **before** sending (roughly `len(text)/4`)
2. Actual tokens **after** sending
3. How far off your estimate was
4. The cost at $0.30/M input and $2.50/M output
5. What 10,000 such calls would cost per month

Run it on five prompts of very different lengths and produce a table.

**Then answer:** which is more expensive per token, input or output? What does that imply about prompt design?

### Task 2: Handle every failure

Write `safe_ask(prompt)` that returns a useful result no matter what goes wrong:

- Retries up to 3 times on a rate-limit error, waiting longer each time
- Returns a clear message if the key is invalid — not a stack trace
- Warns if the response was cut off by `MAX_TOKENS`
- Warns if it was blocked by `SAFETY`
- Never crashes the program

Test it by deliberately using a wrong key, and by setting `max_output_tokens=5`.

> **This is the difference between a notebook cell and an application.**

---

# Concept 2 — Prompt engineering

🧠 **Analogy: briefing a brilliant new intern.**

Imagine an intern who has read almost every book ever written, works instantly, never tires — and has been at your organisation for exactly zero minutes. They do not know who you are, who the work is for, or what "good" looks like here.

**Briefing A:** *"Write something about our product."* → something bland and generic. That is not the intern being unhelpful. **That is you not briefing them.**

**Briefing B:** *"You're writing for our website. The readers are parents of school-age children who are not technical. Explain what our app does and why it's safe. Friendly tone, under 100 words, no jargon, end with a call to action."* → now they can do the job.

**Every complaint about AI output — "too generic", "wrong tone", "too long" — is almost always a briefing problem, not a model problem.**

### The five parts

| Part | Question it answers |
|---|---|
| **Role** | Who should the model be? |
| **Task** | What exactly should it do? |
| **Context** | What does it need to know? |
| **Constraints** | What are the limits? |
| **Format** | What should the output look like? |

## 📘 Examples

### Example 1 — weak vs strong, run side by side

In [ ]:
WEAK = "Write about machine learning."

STRONG = """You are a teacher explaining to first-year students who know
basic Python but no statistics.

Explain what overfitting is.

Use one everyday analogy and one concrete example.
Keep it under 150 words. Do not use the word "variance".

Return two short paragraphs, no headings."""

for label, prompt in [("WEAK", WEAK), ("STRONG", STRONG)]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(ask(prompt))
    print()

**✅ Put the two outputs side by side.** Which one could you use *without editing*?

That difference took two extra minutes of writing. **This is the highest-return skill in the whole of Day 4.**

### Example 2 — the four prompt types on one task

In [ ]:
# ZERO-SHOT: task and data, no examples.
# Watch the FORMAT - it often arrives wrapped in conversational filler.

zero_shot = """Extract the name, occupation, and city from the following
sentence and output it as JSON:

'My name is Sarah, I work as a mechanical engineer, and I just moved to Seattle.'
"""

print("--- ZERO-SHOT ---")
print(ask(zero_shot))

In [ ]:
# ONE-SHOT: a single worked example fixes the format completely.

one_shot = """Extract the flight details into a pipe-separated format.

Input: "I'm flying on Delta flight 402 from JFK to LAX on Tuesday."
Output: Delta | 402 | JFK | LAX | Tuesday

Input: "Book me on United 88 departing from ORD and arriving at SFO tomorrow."
Output:"""

print("--- ONE-SHOT ---")
print(ask(one_shot))

In [ ]:
# FEW-SHOT: several examples teach a pattern that is hard to describe in words.

few_shot = """Classify the customer support ticket as [BILLING], [TECH_ISSUE], or [SALES].

Ticket: "My screen is cracked and the touch sensor won't work."
Category: [TECH_ISSUE]

Ticket: "Do you offer enterprise discounts for teams of 50 or more?"
Category: [SALES]

Ticket: "I was double-charged on my credit card this month."
Category: [BILLING]

Ticket: "How do I upgrade my account from basic to premium?"
Category:"""

print("--- FEW-SHOT ---")
print(ask(few_shot))

In [ ]:
# CHAIN-OF-THOUGHT: ask for reasoning BEFORE the answer.
# Run it BOTH ways. Without the instruction, models frequently answer
# "100 minutes" - the same knee-jerk error humans make.
# The correct answer is 5 minutes.

puzzle = ("If it takes 5 machines 5 minutes to make 5 widgets, "
          "how long would it take 100 machines to make 100 widgets?")

print("--- WITHOUT chain-of-thought ---")
print(ask(puzzle + " Answer with just the number of minutes."))

print("\n--- WITH chain-of-thought ---")
print(ask("Solve this. Break down your reasoning step-by-step before "
          "giving the final answer.\n\n" + puzzle))

### 🧠 Choosing a type — teaching someone your filing system

| Type | How you would explain it to a person | Use when |
|---|---|---|
| **Zero-shot** | "File these by date." | Simple task, flexible format |
| **One-shot** | "File these like this one." | The format must be exact |
| **Few-shot** | "Here are four already filed. See the pattern?" | Classification, or a hard-to-describe pattern |
| **Chain-of-thought** | "Work through it out loud so I can follow." | Maths, logic, many constraints |

> **More examples is not automatically better, and chain-of-thought is not automatically better.** Chain-of-thought gives the best *reasoning* and the worst *format* — it writes a paragraph when you wanted a label. **Match the type to what the task needs.**

### Example 3 — system instructions: a standing brief

In [ ]:
# A system instruction shapes EVERY reply, without being repeated
# in each prompt. It is the intern's job description, not today's task.

tutor = ("You are a statistics tutor for students who have never studied "
         "statistics. Never use jargon without defining it in the same "
         "sentence. Always give one concrete example. Under 120 words.")

for question in ["What is a p-value?", "What is standard deviation?"]:
    print(f"Q: {question}")
    print(ask(question, system_instruction=tutor, temperature=0.3))
    print("-" * 60)

## ✏️ Practice now — Prompting

Five short exercises. Do them **before** moving on — this is where the learning happens.

**1.** Take the weak prompt "Give me some interview questions" and rewrite it with all five parts. Run both.

**2.** Write a **zero-shot** prompt that classifies a sentence as Positive, Negative or Suggestion. Test it on: *"The lab sessions were fine but three hours is too long."*

**3.** Convert it to **few-shot** with three examples. Does the output format change?

**4.** Add a system instruction making the model answer only in bullet points. Test on two questions.

**5.** Write a chain-of-thought prompt for: *"A shop sells pens at ₹12. If I buy 7 and pay with ₹100, what change do I get?"* Does it show its working?

In [ ]:
# 1.


# 2.


# 3.


# 4.


# 5.

<details>
<summary><b>Solutions — open only after you have tried all five</b></summary>

```python
# 1
weak = "Give me some interview questions."
strong = """You are a technical interviewer at a mid-size analytics company.

Write 8 interview questions for a final-year engineering student applying
for a junior data analyst internship.

The student knows Python and Pandas but has never worked on a real project.
The interview is 30 minutes.

Do not ask questions needing deep statistics or production experience.
Mix 4 conceptual with 4 practical questions.

Return a numbered list. After each question add one line starting
with "Looking for:" describing a good answer."""
print(ask(weak)[:200]); print("=" * 40); print(ask(strong))

# 2
comment = "The lab sessions were fine but three hours is too long."
print(ask(f"Classify as Positive, Negative or Suggestion:\n{comment}"))

# 3
few = f"""Classify as Positive, Negative or Suggestion.

Comment: "Great explanation, very clear."
Label: Positive

Comment: "The audio kept cutting out."
Label: Negative

Comment: "Maybe add more examples next time."
Label: Suggestion

Comment: "{comment}"
Label:"""
print(ask(few))     # bare label, no conversational filler

# 4
print(ask("What is recall?", system_instruction="Answer only in bullet points."))

# 5
print(ask("A shop sells pens at 12 rupees. If I buy 7 and pay with 100 rupees, "
          "what change do I get? Think step by step before answering."))
```

</details>

## 🧠 Quick quiz — Prompting

Answer all five from memory. No scrolling back.

**Q1.** Name the five parts of a good prompt.

**Q2.** You need output in an exact fixed format. Which prompt type, and why?

**Q3.** When is chain-of-thought the wrong choice?

**Q4.** What is the difference between a system instruction and putting the same text in the prompt?

**Q5.** Your colleague says "the AI gives bad answers". What is your first question?

<details>
<summary><b>Answers</b></summary>

**A1.** **Role** (who the model should be), **Task** (what to do), **Context** (what it needs to know), **Constraints** (the limits), **Format** (what the output should look like).

**A2.** **One-shot.** A single worked example sets a strict template far more reliably than describing the format in words, and it removes the conversational filler that zero-shot tends to add.

**A3.** When you need a clean parseable output. It produces the best *reasoning* and the worst *format* — a paragraph when you wanted a label. If you need both, ask for the reasoning then a final line like `ANSWER: <x>`.

**A4.** A system instruction is a **standing** brief that applies to every turn without being repeated. It is the intern's job description rather than today's task — and it keeps your per-call prompts shorter, which costs less.

**A5.** **"Can I see your prompt?"** Roughly nine times in ten the output is generic because the briefing was generic — no role, no audience, no constraints, no format. It is a briefing problem, not a model problem.

</details>

## 🎯 Tasks — Prompting

Longer work. Do these after the session, in your own time.

### Task 1: The prompt portfolio

Build five prompts that solve problems **you actually have** — summarising your notes, drafting a message to a professor, generating practice questions, explaining a topic to a friend.

For each, record:
- the weak one-line version
- the strong five-part version
- both outputs
- which prompt type you chose and **why**
- one guardrail line, and what it prevents

**Then use the strong prompts this week.** A prompt you saved and reuse is worth more than fifty you tried once.

### Task 2: All four types, one task, scored

Take **one** classification task and write it four ways. Test all four on this deliberately awkward input:

```text
"The lab sessions were fine but honestly three hours is too long,
maybe split it into two."
```

| Version | Label given | Filler around it? | Usable in code? |
|---|---|---|---|
| Zero-shot | | | |
| One-shot | | | |
| Few-shot | | | |
| Chain-of-thought | | | |

**What you should find:** zero-shot wraps the answer in chat; one-shot and few-shot return the bare label; chain-of-thought gives the best reasoning and the worst format.

**And notice:** the comment is genuinely ambiguous — positive *and* a suggestion. Different prompt types may disagree. **When humans would disagree on a label, models disagree too.**

### Task 3: Prompt golf

Get the model to output **exactly** the phrase `The eagle flies at midnight` — with no extra text — **without using the words `eagle`, `flies`, or `midnight` in your prompt.**

Then make your prompt as short as possible.

Three approaches people discover:
- **Semantic:** describe each word ("America's bald national animal")
- **Cultural:** ask for the cliché spy password
- **Linguistic:** ask for a translation from another language

**The lesson:** getting a model to drop its conversational filler requires explicit format constraints — a crucial skill when you need raw data back from an API.

---

# Concept 3 — Temperature, top-p and top-k

🧠 **Analogy: the spice dial and the spice shelf.**

- **`temperature`** is the dial. 0 = plain, safe, identical every time. 1 = bold and varied.
- **`top_k` / `top_p`** decide **which spices are on the shelf at all**, before the dial is even touched.

```text
STEP 1 - decide which candidates are ALLOWED
         top_k = 40   -> keep only the 40 most likely tokens
         top_p = 0.9  -> keep the most likely tokens until their
                         probabilities add up to 90%, drop the rest

STEP 2 - decide how ADVENTUROUSLY to pick from that shelf
         temperature  -> 0 = always the top one
                         1 = sample freely across the shelf
```

**`top_k` and `top_p` decide who is invited. `temperature` decides who gets chosen.** That ordering is the whole lesson.

## 📘 Examples

### Example 1 — the temperature dial

In [ ]:
PROMPT = "Write a tagline for a coffee shop. Return only the tagline."

for temperature in [0.0, 1.0]:
    print("=" * 55)
    print(f"TEMPERATURE = {temperature}")
    print("=" * 55)
    for run in range(1, 4):
        print(f"  Run {run}: {ask(PROMPT, temperature=temperature, max_output_tokens=50).strip()}")
    print()

**✅ What you should see.** At `0.0` the three runs are identical or nearly so — the model always takes the single most probable token, so there is no randomness left to produce a difference. At `1.0` they differ noticeably.

**High temperature gives you the highest highs and the lowest lows.** Low temperature gives you consistently acceptable and consistently dull.

### Example 2 — top_k in the extreme

In [ ]:
# The fastest way to feel what top_k does is to set it to 1.
# temperature stays at 1.0 for BOTH runs.

for label, config in {
    "top_k=1  (only ONE candidate allowed)": dict(temperature=1.0, top_k=1),
    "top_k=40 (forty candidates allowed)":  dict(temperature=1.0, top_k=40),
}.items():
    print(f"--- {label} ---")
    for run in range(3):
        print("   ", ask("Suggest a name for a new coffee shop. Reply with the name only.",
                         max_output_tokens=20, **config).strip())
    print()

**✅ With `top_k=1` the runs are identical even though temperature is 1.0. Why?**

Because temperature decides *how adventurously to choose among the candidates*, and with one candidate there is nothing to choose between.

> **A high temperature cannot create variety that `top_k` has already removed.** That is the relationship between the two dials, and it is much easier to feel than to read.

### Example 3 — settings for real jobs

In [ ]:
settings = {
    "Factual / extraction": dict(temperature=0.0, top_p=0.1, top_k=1),
    "Balanced / support":   dict(temperature=0.4, top_p=0.9, top_k=40),
    "Creative / naming":    dict(temperature=1.2, top_p=1.0, top_k=100),
}

for name, config in settings.items():
    print(f"--- {name}  {config} ---")
    print("   ", ask("Describe a rainy evening in one sentence.",
                     max_output_tokens=60, **config).strip())
    print()

### 📊 How to choose your settings

| Task | temperature | Why |
|---|---|---|
| Extracting dates from invoices | **0.0** | The same input must give the same output, every time |
| Translating a legal notice | **0.0–0.2** | Accuracy is everything; invention is dangerous |
| A customer support reply | **0.3–0.5** | Mostly consistent, slightly natural |
| Generating quiz questions | **0.3–0.5** | Mostly factual, but you do not want the same five every run |
| Brainstorming campaign slogans | **0.9–1.2** | Variety is the entire point |
| Writing a poem | **0.8–1.0** | A predictable poem is a bad poem |

> ⚠️ **Change `temperature` first and leave the other two at their defaults.** Adjust all three at once and you will never know which one caused the change you are looking at. This is the same discipline as changing one variable in an experiment.

## ✏️ Practice now — Sampling parameters

Five short exercises. Do them **before** moving on — this is where the learning happens.

**1.** Run the same prompt 5 times at `temperature=0.0`. Are all five identical?

**2.** Run it 5 times at `temperature=1.5`. Count how many are distinct.

**3.** Set `temperature=1.0, top_k=1` and run 3 times. Explain the result in one sentence.

**4.** Set `temperature=0.0, top_k=100` and run 3 times. Does the high top_k create variety? Why not?

**5.** For each of these, choose a temperature and justify it: (a) extracting invoice totals, (b) naming a college fest, (c) summarising a medical report.

In [ ]:
P = "Suggest a tagline for a bookshop. Reply with the tagline only."

# 1.


# 2.


# 3.


# 4.


# 5.

<details>
<summary><b>Solutions — open only after you have tried all five</b></summary>

```python
P = "Suggest a tagline for a bookshop. Reply with the tagline only."

# 1
print("T=0.0:")
for _ in range(5): print("  ", ask(P, temperature=0.0, max_output_tokens=30).strip())

# 2
print("\nT=1.5:")
outs = [ask(P, temperature=1.5, max_output_tokens=30).strip() for _ in range(5)]
for o in outs: print("  ", o)
print("distinct:", len(set(outs)))

# 3
print("\nT=1.0, top_k=1:")
for _ in range(3): print("  ", ask(P, temperature=1.0, top_k=1, max_output_tokens=30).strip())
# Identical: with one candidate there is nothing for temperature to choose between.

# 4
print("\nT=0.0, top_k=100:")
for _ in range(3): print("  ", ask(P, temperature=0.0, top_k=100, max_output_tokens=30).strip())
# Still identical: top_k widened the shelf, but temperature 0 always takes the top item.

# 5  (a) 0.0 - must be reproducible
#    (b) 1.0 - variety is the point
#    (c) 0.0 - invention in a medical summary is dangerous
```

</details>

## 🧠 Quick quiz — Sampling parameters

Answer all five from memory. No scrolling back.

**Q1.** What does `temperature=0` actually do to the probability distribution?

**Q2.** `temperature=1.0` but `top_k=1`. Will three runs differ? Why?

**Q3.** `temperature=0.0` but `top_k=100`. Will three runs differ?

**Q4.** What is the difference between `top_k` and `top_p`?

**Q5.** Why should you change only one sampling parameter at a time?

<details>
<summary><b>Answers</b></summary>

**A1.** It collapses it: the single most probable token gets probability 1.0 and everything else gets 0. There is no randomness left, so the same prompt gives the same output every time.

**A2.** **No — identical.** `top_k=1` allows only one candidate token, and temperature only controls how adventurously you choose *among* candidates. A high temperature cannot create variety that `top_k` already removed.

**A3.** **No.** A wide shelf does not matter if the dial always picks the single most likely item. Both parameters have to allow variety before you see any.

**A4.** `top_k` keeps a **fixed number** of candidates (the 40 most likely). `top_p` keeps however many are needed for their probabilities to sum to *p* — so it adapts: few candidates when the model is confident, many when it is unsure.

**A5.** Because if you change all three and the output changes, you have learned nothing about which one caused it. It is the same discipline as changing one variable in an experiment — and the same reason this course keeps saying 'change one thing'.

</details>

## 🎯 Tasks — Sampling parameters

Longer work. Do these after the session, in your own time.

### Task 1: The temperature study

Pick one prompt and run it **five times** at each of: 0.0, 0.3, 0.7, 1.0, 1.5.

Build a table: temperature, the five outputs, how many were distinct, and your rating of the best output at that setting (1–5).

Then answer:
1. At which temperature did outputs first start to differ?
2. At which did quality start to *drop*?
3. **Where is the sweet spot for this particular task, and would it be the same for a different task?**

### Task 2: Reproducibility audit

You are shipping an app that extracts invoice totals. It must give the same answer for the same invoice, every time.

1. Write the extraction prompt
2. Run it 10 times on the same input at default settings. Count distinct outputs
3. Now lock the settings down to make it deterministic
4. Run 10 more times. Confirm all identical
5. Write two sentences for your README explaining **why** those settings were chosen

> An app that gives a different answer to the same question is not a bug you can debug — it is a design choice you failed to make.

---

# Concept 4 — Structured output and conversation memory

Two things that turn a demo into an application.

**Structured output** — your code needs `data["city"]`, not a paragraph that happens to mention Seattle.

**Memory** — the API is stateless, so if you want a conversation, *your program* must provide it.

## 📘 Examples

### Example 1 — asking for JSON is not enough; enforce it

In [ ]:
# WITHOUT enforcement: you often get "Here is your JSON:" wrapped around it,
# and json.loads() then fails.
loose = ask("Extract name, occupation and city as JSON: "
            "'My name is Sarah, I work as a mechanical engineer, "
            "and I just moved to Seattle.'")
print("--- asked politely ---")
print(repr(loose[:200]))

In [ ]:
# WITH enforcement: response_mime_type forces valid JSON.
strict = ask("Extract name, occupation and city as JSON: "
             "'My name is Sarah, I work as a mechanical engineer, "
             "and I just moved to Seattle.'",
             response_mime_type="application/json",
             temperature=0.0)

data = json.loads(strict)          # parses cleanly, every time

print("--- enforced ---")
print(type(data).__name__, data)
print("\nNow you can do this in code:")
print("  city ->", data.get("city"))

> **This is the difference between a demo and an application.** Without `response_mime_type`, `json.loads` fails perhaps one time in five, and your app crashes in front of a user.

### Example 2 — a schema the model must follow

In [ ]:
# Describing the exact shape you want makes the output predictable enough
# to build a UI on.

prompt = """Create a 3-question multiple-choice quiz on overfitting.

Return ONLY valid JSON in this exact shape:
{
  "questions": [
    {
      "question": "the question text",
      "options": ["A", "B", "C", "D"],
      "correct_index": 0,
      "explanation": "why that answer is correct, one sentence"
    }
  ]
}

correct_index is a 0-based integer into the options array.
Vary which position the correct answer sits in."""

quiz_data = json.loads(ask(prompt, response_mime_type="application/json",
                           temperature=0.7))

for i, q in enumerate(quiz_data["questions"], 1):
    print(f"Q{i}. {q['question']}")
    for j, opt in enumerate(q["options"]):
        marker = "✓" if j == q["correct_index"] else " "
        print(f"   {marker} {chr(65 + j)}. {opt}")
    print(f"   -> {q['explanation']}\n")

### Example 3 — prove the model has no memory, then give it some

In [ ]:
# PART 1: two INDEPENDENT calls. The model has no idea what "it" means.

print("=== WITHOUT memory (two separate calls) ===")
print("Q1: What is Python?")
print("A1:", ask("What is Python?")[:140], "...\n")
print("Q2: Who created it?")
print("A2:", ask("Who created it?")[:200])

In [ ]:
# PART 2: the SAME two questions, in a chat that keeps history.

print("=== WITH memory (a chat session) ===")
chat = client.chats.create(model=MODEL_NAME)

print("Q1: What is Python?")
print("A1:", chat.send_message("What is Python?").text[:140], "...\n")
print("Q2: Who created it?")
print("A2:", chat.send_message("Who created it?").text[:200])

**✅ Nothing about the model changed between the two parts.**

The only difference is that `chats.create()` re-sends the earlier messages with every request.

> **Memory is not a property of the model. It is a feature your program provides.** Every chatbot you have ever used works exactly this way.

**🔁 Change one thing:** keep the chat going for ten more turns, then ask about something from the first message. Eventually the history exceeds the **context window** and the earliest messages fall out. Real chatbots handle this by *summarising* old messages rather than sending them all.

### Example 4 — grounding: making the model refuse to guess

In [ ]:
# The single most useful prompt pattern in production.
# This is RAG without the vector database.

DOCUMENT = """
The internship programme runs for four weeks. Week 1 is five days of
classroom training covering Python, machine learning, and generative AI.
Weeks 2 to 4 are the capstone project phase, with online reviews at the
end of each week. Students must submit a repository and a report by the
end of week 4. Attendance in week 1 is compulsory. The pass mark is 50 percent.
"""

REFUSAL = "Not stated in the provided document."

def grounded_answer(question):
    prompt = f"""Answer the question using ONLY the document below.

If the answer is not in the document, reply with exactly this sentence
and nothing else: "{REFUSAL}"

Do not use outside knowledge. Do not guess. Do not infer beyond what is written.

DOCUMENT:
\"\"\"{DOCUMENT}\"\"\"

QUESTION: {question}"""
    return ask(prompt, temperature=0.0).strip()

for question in ["How long is the programme?",
                 "What is the pass mark?",
                 "Who is the instructor?",        # NOT in the document
                 "What is the course fee?"]:      # NOT in the document
    answer = grounded_answer(question)
    mark = "REFUSED ✅" if REFUSAL.lower() in answer.lower() else "answered"
    print(f"[{mark}] {question}\n    {answer}\n")

**✅ The last two questions should be refused.** A model that answers them has hallucinated.

**Grounding is not a setting you enable — it is a sentence you write in the prompt**, and it works because you told the model what it is *not allowed* to do.

**🔁 Change one thing:** delete the line `Do not use outside knowledge` and ask again. Watch it start inventing an instructor.

## ✏️ Practice now — Structured output and memory

Five short exercises. Do them **before** moving on — this is where the learning happens.

**1.** Ask for a JSON object with keys `people`, `places`, `dates` from any paragraph. Parse it with `json.loads`.

**2.** Run the same extraction 5 times **without** `response_mime_type`. How many times does `json.loads` fail?

**3.** Build a `chats.create()` session and ask three linked questions where each depends on the last.

**4.** Use `grounded_answer` with your own short document. Ask two answerable and two unanswerable questions.

**5.** Remove the refusal instruction from the grounding prompt and ask an unanswerable question. What changes?

In [ ]:
# 1.


# 2.


# 3.


# 4.


# 5.

<details>
<summary><b>Solutions — open only after you have tried all five</b></summary>

```python
# 1
para = "Dr Meera Nair visited Chennai on 3 March 2024 and met Ravi Kumar in Bangalore."
d = json.loads(ask(f"Extract people, places and dates as JSON with keys "
                   f"'people','places','dates' (each a list): {para}",
                   response_mime_type="application/json", temperature=0.0))
print(d)

# 2
fails = 0
for _ in range(5):
    try:
        json.loads(ask(f"Extract people, places and dates as JSON: {para}"))
    except json.JSONDecodeError:
        fails += 1
print("json.loads failed", fails, "times out of 5 without enforcement")

# 3
c = client.chats.create(model=MODEL_NAME)
print(c.send_message("Name a famous Indian mathematician.").text[:120])
print(c.send_message("What was his most famous contribution?").text[:160])
print(c.send_message("Explain that to a school student.").text[:200])

# 4  Replace DOCUMENT with your own text and re-run grounded_answer.

# 5  Without the refusal line the model invents a plausible answer instead
#    of admitting the document does not contain one. That invention is the
#    hallucination, and the single prompt line is what prevented it.
```

</details>

## 🧠 Quick quiz — Structured output and memory

Answer all five from memory. No scrolling back.

**Q1.** Why is `response_mime_type="application/json"` better than just asking for JSON in the prompt?

**Q2.** Your chatbot forgets the previous question. Why, and where is the fix?

**Q3.** A long conversation starts losing early details. What is happening?

**Q4.** What is grounding, and how do you actually do it?

**Q5.** Your grounded app refuses to answer a question. Is that a bug?

<details>
<summary><b>Answers</b></summary>

**A1.** Asking politely still leaves conversational filler like 'Here is your JSON:' perhaps one time in five, and `json.loads` then throws. Enforcing the type guarantees valid JSON, so your app does not crash in front of a user.

**A2.** The API is **stateless** — each call starts blank. The fix is in **your program**: re-send the conversation history with every request, which is what `client.chats.create()` does for you.

**A3.** The history has exceeded the **context window**, so the earliest messages fall out. Production chatbots handle this by summarising old turns instead of sending them verbatim — which keeps the meaning while cutting the tokens.

**A4.** Giving the model the source text and forbidding it from using anything else. You do it with prompt lines: 'Answer using ONLY the document below', 'Do not use outside knowledge', and an exact refusal sentence for when the answer is not there.

**A5.** **No — that is the app working correctly.** A refusal is far more useful than a confident invention. An app that always answers is an app that hallucinates when it does not know.

</details>

## 🎯 Tasks — Structured output and memory

Longer work. Do these after the session, in your own time.

### Task 1: The entity extractor

Build a tool that, from any block of text, returns a JSON object with keys `people`, `places`, `dates`, `organisations` — each a list.

1. Use `response_mime_type` and `temperature=0.0`
2. Run it on **five** different paragraphs
3. Record how often the JSON parsed first time
4. Add validation: what happens if a key is missing? Handle it
5. Add a `confidence` field: `"high"` if stated explicitly, `"low"` if inferred

**Then test it on a paragraph containing no entities at all.** Does it return empty lists, or invent something?

### Task 2: The grounded document assistant

Build a question-answering tool over a document of your choice (your lecture notes work well).

1. Load the document
2. Implement `grounded_answer` with a refusal sentence
3. Make it also quote the supporting sentence, prefixed `SOURCE:`
4. Test with **three** answerable and **three** unanswerable questions
5. **Report how many times it correctly refused**
6. Report the token cost per question

**Then the upgrade:** split long documents into 500-word chunks and send only the chunks most relevant to the question (keyword overlap is fine). Report how many tokens you saved.

**You have now built RAG.**

### Task 3: Conversation memory that scales

A chatbot that resends the whole history gets more expensive with every turn. Fix that.

1. Build a chat loop that tracks total tokens per turn
2. Plot tokens against turn number. What shape is it?
3. Implement a cap: keep only the last 10 messages
4. Implement something better: **summarise** older messages into one short paragraph and prepend that
5. Compare the token curves of all three approaches

**Answer:** what does the summarising approach lose, and when would that matter?

---
# ✅ Day 4 exit task

You should be able to show:

1. A working Gemini API call from your own environment
2. The temperature comparison table, filled with your real outputs
3. A weak prompt and a strong five-part prompt for a task **you actually care about**, run side by side
4. A grounded prompt that correctly **refuses** a question its document cannot answer
5. A one-sentence answer to **"Why does an LLM hallucinate?"** — in terms of how it works, not "it makes mistakes"

In [ ]:
# Your exit task working here.

---
## What next

| | |
|---|---|
| **Previous** | [Day 4a — Clustering and how LLMs work](04_day4a_clustering.ipynb) |
| **Next** | [Day 5 — Hugging Face and open-source models](06_day5_huggingface.ipynb) |
| **Build apps** | [Streamlit Apps Collection](../tutorials/apps/streamlit-apps-collection.md) — 15 runnable apps |
| **Prompts** | [prompts.md](../prompts.md) — every demo prompt, ready to copy |
| **More practice** | [exercises-assignments.md](../exercises-assignments.md) |